# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

Import Library

In [1]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Get Dataset and Change Path

In [3]:
import kagglehub
import shutil
import os

# Download dataset
path = kagglehub.dataset_download(
    "albertnathaniel12/indonesian-spices-dataset"
)

print("Path dataset asli:", path)

# Direktori tujuan di MacBook
destination = os.path.join(os.getcwd(), "data")

# Buat parent directory jika belum tersedia
os.makedirs(os.path.dirname(destination), exist_ok=True)

# Hapus folder lama jika sudah ada
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy dataset
shutil.copytree(path, destination)

print("Dataset berhasil disalin ke:", destination)
print("Isi dataset:", os.listdir(destination))

Path dataset asli: /Users/pangeran/.cache/kagglehub/datasets/albertnathaniel12/indonesian-spices-dataset/versions/11
Dataset berhasil disalin ke: /Users/pangeran/Documents/Telkom University/semester 5/visi komputer/TugasCNN/IndonesiaSpicesProject_CNN/data
Isi dataset: ['Indonesian Spices Dataset']


## Split Dataset: Train / Validation / Test

Dataset asli berisi 31 kelas rempah dengan struktur `data/Indonesian Spices Dataset/<nama kelas>/*.jpg`.

Split dilakukan secara **stratified** (proporsi tiap kelas sama di semua subset) dengan rasio **70% train / 15% validation / 15% test**.
Hasilnya ditulis ke `data_split/` dengan format `ImageFolder` sehingga langsung bisa dipakai `torchvision.datasets.ImageFolder`.

> Catatan: folder output sengaja dipisah dari `data/`, karena cell download di atas menghapus isi `data/` setiap kali dijalankan.

In [4]:
from pathlib import Path
import shutil
import random

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Path relatif: folder tempat notebook ini dijalankan (folder project)
PROJECT_DIR = Path.cwd()
SOURCE_DIR = PROJECT_DIR / "data" / "Indonesian Spices Dataset"
OUTPUT_DIR = PROJECT_DIR / "data_split"

TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
SEED = 42
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Total rasio harus 1.0"
assert SOURCE_DIR.exists(), f"Dataset tidak ditemukan di {SOURCE_DIR}. Jalankan cell download di atas."

random.seed(SEED)

# Kumpulkan semua file gambar beserta labelnya
records = []
for class_dir in sorted(p for p in SOURCE_DIR.iterdir() if p.is_dir()):
    for img_path in sorted(class_dir.rglob("*")):
        if img_path.is_file() and img_path.suffix.lower() in IMG_EXTENSIONS:
            records.append({"filepath": img_path, "label": class_dir.name})

df = pd.DataFrame(records)

print(f"Project dir  : {PROJECT_DIR}")
print(f"Jumlah kelas : {df['label'].nunique()}")
print(f"Jumlah gambar: {len(df)}")
print("\nDistribusi per kelas:")
print(df["label"].value_counts().sort_index())

Project dir  : /Users/pangeran/Documents/Telkom University/semester 5/visi komputer/TugasCNN/IndonesiaSpicesProject_CNN
Jumlah kelas : 31
Jumlah gambar: 6510

Distribusi per kelas:
label
adas             210
andaliman        210
asam jawa        210
bawang bombai    210
bawang merah     210
bawang putih     210
biji ketumbar    210
bukan rempah     210
bunga lawang     210
cengkeh          210
daun jeruk       210
daun kemangi     210
daun ketumbar    210
daun salam       210
jahe             210
jinten           210
kapulaga         210
kayu manis       210
kayu secang      210
kemiri           210
kemukus          210
kencur           210
kluwek           210
kunyit           210
lada             210
lengkuas         210
pala             210
saffron          210
serai            210
vanili           210
wijen            210
Name: count, dtype: int64


## Deduplikasi (sebelum split)

EDA tahap 1 menemukan 144 grup file byte-identik dan 440 grup near-duplicate, dengan 68 grup di antaranya
tersebar lintas split. Duplikat yang muncul di train sekaligus di test membuat akurasi test **terlihat lebih
tinggi dari kemampuan model sebenarnya**, jadi duplikat harus dibuang **sebelum** split — bukan sesudahnya.

Dua level deteksi:

1. **`md5`** — file byte-identik. Tidak ada ruang salah.
2. **`dhash`** (difference hash) — menangkap gambar yang sama walau sudah di-resize atau di-encode ulang.

`dhash` sendirian terlalu agresif di dataset ini: foto objek kecil di latar putih punya hash yang sama
walau isinya beda (mis. semangkuk adas vs cengkeh di talenan). Karena itu setiap grup `dhash`
**diverifikasi ulang di level piksel** — RMSE thumbnail 32×32 grayscale harus < 5; grup yang gagal
verifikasi dianggap false positive dan semua anggotanya dipertahankan.

Grup duplikat yang anggotanya punya **label berbeda** berarti ground truth-nya ambigu
(satu di antaranya salah label), jadi seluruh grup dibuang.

In [5]:
import hashlib
import numpy as np
from PIL import Image

DHASH_SIZE = 8
THUMB_SIZE = 32
RMSE_THRESHOLD = 5.0   # verifikasi piksel untuk grup dhash; di bawah ini = benar-benar duplikat


def image_signature(path):
    """Kembalikan (md5, dhash, thumbnail 32x32 grayscale) untuk satu file."""
    raw = path.read_bytes()
    with Image.open(path) as im:
        im.load()
        g = im.convert("L")
        small = g.resize((DHASH_SIZE + 1, DHASH_SIZE), Image.LANCZOS)
        a = np.asarray(small, dtype=np.int16)
        dh = np.packbits((a[:, 1:] > a[:, :-1]).flatten()).tobytes().hex()
        thumb = np.asarray(g.resize((THUMB_SIZE, THUMB_SIZE), Image.BILINEAR), dtype=np.float32)
    return hashlib.md5(raw).hexdigest(), dh, thumb


sigs = [image_signature(p) for p in tqdm(df["filepath"], desc="hash citra")]
df["md5"] = [s[0] for s in sigs]
df["dhash"] = [s[1] for s in sigs]
thumbs = {p: s[2] for p, s in zip(df["filepath"], sigs)}


def max_rmse(paths):
    """Jarak piksel terbesar antar anggota grup; 0 berarti identik."""
    t = [thumbs[p] for p in paths]
    return max(
        (float(np.sqrt(((t[i] - t[j]) ** 2).mean())) for i in range(len(t)) for j in range(i + 1, len(t))),
        default=0.0,
    )


print(f"Sebelum dedup: {len(df)} citra, {df['label'].nunique()} kelas")
print(f"File byte-identik (md5) : {df.duplicated('md5', keep=False).sum()}")
print(f"Kandidat near-dup (dhash): {df.duplicated('dhash', keep=False).sum()}")

hash citra:   0%|          | 0/6510 [00:00<?, ?it/s]

/opt/homebrew/anaconda3/lib/python3.13/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Sebelum dedup: 6510 citra, 31 kelas
File byte-identik (md5) : 293
Kandidat near-dup (dhash): 929


In [6]:
# Grup duplikat = anggota dhash yang sama DAN lolos verifikasi piksel.
# md5 identik otomatis lolos (RMSE-nya pasti 0), jadi dhash cukup sebagai kunci grup.
drop_duplicate, drop_conflict, false_positive_groups = [], [], []

for dh, grp in df[df.duplicated("dhash", keep=False)].groupby("dhash"):
    paths = list(grp["filepath"])
    if max_rmse(paths) >= RMSE_THRESHOLD:
        false_positive_groups.append((dh, sorted(set(grp["label"])), max_rmse(paths)))
        continue

    if grp["label"].nunique() > 1:
        # Gambar sama tapi labelnya beda -> ground truth ambigu, buang seluruh grup
        drop_conflict.extend(paths)
    else:
        # Simpan satu wakil (file pertama secara urutan), buang sisanya
        drop_duplicate.extend(sorted(paths)[1:])

print(f"Grup lolos verifikasi, dibuang sebagai duplikat : {len(drop_duplicate)} file")
print(f"Grup berlabel konflik, dibuang seluruhnya       : {len(drop_conflict)} file")
print(f"Grup ditolak verifikasi piksel (false positive) : {len(false_positive_groups)} grup, dipertahankan")

if false_positive_groups:
    print("\nDetail false positive:")
    for dh, labels, d in false_positive_groups:
        print(f"  RMSE {d:5.2f}  label: {', '.join(labels)}")

if drop_conflict:
    print("\nFile dengan label konflik yang dibuang:")
    for p in sorted(drop_conflict):
        print(f"  {p.parent.name}/{p.name}")

Grup lolos verifikasi, dibuang sebagai duplikat : 477 file
Grup berlabel konflik, dibuang seluruhnya       : 5 file
Grup ditolak verifikasi piksel (false positive) : 8 grup, dipertahankan

Detail false positive:
  RMSE  7.30  label: adas, cengkeh
  RMSE  7.45  label: wijen
  RMSE 12.79  label: daun ketumbar
  RMSE 14.08  label: kayu manis
  RMSE  7.77  label: kluwek
  RMSE  6.20  label: wijen
  RMSE 11.01  label: kunyit
  RMSE 16.86  label: lengkuas

File dengan label konflik yang dibuang:
  biji ketumbar/biji-ketumbar_91.jpg
  daun jeruk/daun-jeruk_140.jpg
  daun jeruk/daun-jeruk_153.jpg
  daun salam/daun-salam_208.jpg
  lada/lada_89.jpg


In [7]:
to_drop = set(drop_duplicate) | set(drop_conflict)
df_raw = df.copy()
df = df[~df["filepath"].isin(to_drop)].reset_index(drop=True)

removed = len(df_raw) - len(df)
print(f"Dibuang : {removed} citra ({removed / len(df_raw):.2%})")
print(f"Tersisa : {len(df)} citra, {df['label'].nunique()} kelas")

# Verifikasi: tidak boleh ada sisa duplikat yang lolos verifikasi piksel
sisa = [
    dh for dh, grp in df[df.duplicated("dhash", keep=False)].groupby("dhash")
    if max_rmse(list(grp["filepath"])) < RMSE_THRESHOLD
]
print(f"Sisa duplikat terverifikasi: {len(sisa)} grup")
assert not sisa, "Masih ada duplikat tersisa"

before = df_raw["label"].value_counts()
after = df["label"].value_counts()
delta = (before - after.reindex(before.index).fillna(0)).astype(int).sort_values(ascending=False)
print("\nCitra terbuang per kelas (10 teratas):")
print(delta.head(10).to_string())
print(f"\nDistribusi akhir: min {after.min()}, max {after.max()}, "
      f"imbalance ratio {after.max() / after.min():.2f}x")

# Syarat tugas harus tetap terpenuhi setelah dedup
assert df["label"].nunique() >= 5 and len(df) >= 1000

Dibuang : 482 citra (7.40%)
Tersisa : 6028 citra, 31 kelas
Sisa duplikat terverifikasi: 0 grup

Citra terbuang per kelas (10 teratas):
label
asam jawa        89
kayu secang      55
jinten           38
cengkeh          28
biji ketumbar    22
daun jeruk       22
andaliman        21
daun salam       20
bunga lawang     19
serai            17

Distribusi akhir: min 121, max 209, imbalance ratio 1.73x


### Stratified split 70 / 15 / 15

Dilakukan dua tahap: pertama memisahkan train vs sisa, lalu sisa dibagi menjadi validation dan test.

In [8]:
# Tahap 1: train vs (val + test)
train_df, temp_df = train_test_split(
    df,
    test_size=(VAL_RATIO + TEST_RATIO),
    stratify=df["label"],
    random_state=SEED,
    shuffle=True,
)

# Tahap 2: val vs test (proporsional terhadap sisa)
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
    stratify=temp_df["label"],
    random_state=SEED,
    shuffle=True,
)

splits = {"train": train_df, "val": val_df, "test": test_df}

for name, part in splits.items():
    print(f"{name:5s}: {len(part):5d} gambar ({len(part) / len(df):.1%})")

# Pastikan tidak ada file yang bocor antar subset
assert len(train_df) + len(val_df) + len(test_df) == len(df)
assert not (set(train_df['filepath']) & set(val_df['filepath']) & set(test_df['filepath']))

train:  4219 gambar (70.0%)
val  :   904 gambar (15.0%)
test :   905 gambar (15.0%)


### Copy file ke struktur `ImageFolder`

```
data_split/
├── train/<nama kelas>/*.jpg
├── val/<nama kelas>/*.jpg
└── test/<nama kelas>/*.jpg
```

In [9]:
# Mulai dari kondisi bersih agar hasil split tidak bercampur dengan run sebelumnya
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

manifest_rows = []

for split_name, part in splits.items():
    for label in df["label"].unique():
        (OUTPUT_DIR / split_name / label).mkdir(parents=True, exist_ok=True)

    for row in tqdm(part.itertuples(index=False), total=len(part), desc=f"copy {split_name}"):
        dest = OUTPUT_DIR / split_name / row.label / row.filepath.name
        shutil.copy2(row.filepath, dest)
        manifest_rows.append(
            {
                "split": split_name,
                "label": row.label,
                "filename": row.filepath.name,
                "filepath": str(dest.relative_to(PROJECT_DIR)),
            }
        )

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)

print(f"Selesai. Hasil split: {OUTPUT_DIR}")
print(f"Manifest  : {OUTPUT_DIR / 'manifest.csv'} ({len(manifest)} baris)")

copy train:   0%|          | 0/4219 [00:00<?, ?it/s]

copy val:   0%|          | 0/904 [00:00<?, ?it/s]

copy test:   0%|          | 0/905 [00:00<?, ?it/s]

Selesai. Hasil split: /Users/pangeran/Documents/Telkom University/semester 5/visi komputer/TugasCNN/IndonesiaSpicesProject_CNN/data_split
Manifest  : /Users/pangeran/Documents/Telkom University/semester 5/visi komputer/TugasCNN/IndonesiaSpicesProject_CNN/data_split/manifest.csv (6028 baris)


### Verifikasi hasil split

In [10]:
# Hitung ulang langsung dari hasil di disk, bukan dari dataframe
counts = {
    split_name: {
        class_dir.name: sum(1 for f in class_dir.iterdir() if f.is_file())
        for class_dir in sorted((OUTPUT_DIR / split_name).iterdir())
        if class_dir.is_dir()
    }
    for split_name in ("train", "val", "test")
}

summary = pd.DataFrame(counts).fillna(0).astype(int)
summary["total"] = summary.sum(axis=1)

print(summary.to_string())
print("\nTotal per split:")
print(summary[["train", "val", "test"]].sum())
print(f"\nTotal keseluruhan: {summary['total'].sum()} (dataset asli: {len(df)})")

               train  val  test  total
adas             137   29    29    195
andaliman        132   29    28    189
asam jawa         85   18    18    121
bawang bombai    146   31    31    208
bawang merah     143   31    31    205
bawang putih     143   31    31    205
biji ketumbar    132   28    28    188
bukan rempah     146   31    31    208
bunga lawang     134   29    28    191
cengkeh          127   28    27    182
daun jeruk       132   28    28    188
daun kemangi     143   31    31    205
daun ketumbar    136   29    29    194
daun salam       133   29    28    190
jahe             137   30    29    196
jinten           120   26    26    172
kapulaga         141   30    31    202
kayu manis       146   31    32    209
kayu secang      109   23    23    155
kemiri           146   31    31    208
kemukus          143   30    31    204
kencur           141   30    31    202
kluwek           136   29    29    194
kunyit           143   30    31    204
lada             143   31

In [11]:
# Contoh pemakaian dengan torchvision ImageFolder
from torchvision import datasets, transforms

preview_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

for split_name in ("train", "val", "test"):
    ds = datasets.ImageFolder(OUTPUT_DIR / split_name, transform=preview_tf)
    print(f"{split_name:5s}: {len(ds)} gambar, {len(ds.classes)} kelas")

print("\nMapping kelas -> index:")
print(ds.class_to_idx)

train: 4219 gambar, 31 kelas
val  : 904 gambar, 31 kelas
test : 905 gambar, 31 kelas

Mapping kelas -> index:
{'adas': 0, 'andaliman': 1, 'asam jawa': 2, 'bawang bombai': 3, 'bawang merah': 4, 'bawang putih': 5, 'biji ketumbar': 6, 'bukan rempah': 7, 'bunga lawang': 8, 'cengkeh': 9, 'daun jeruk': 10, 'daun kemangi': 11, 'daun ketumbar': 12, 'daun salam': 13, 'jahe': 14, 'jinten': 15, 'kapulaga': 16, 'kayu manis': 17, 'kayu secang': 18, 'kemiri': 19, 'kemukus': 20, 'kencur': 21, 'kluwek': 22, 'kunyit': 23, 'lada': 24, 'lengkuas': 25, 'pala': 26, 'saffron': 27, 'serai': 28, 'vanili': 29, 'wijen': 30}
